## Preliminary BABEL development results

Reproducible presentation figures for mouse gastruloid H3K4me3 → RNA.
Run all cells to load the saved metrics (inside the repository) or the explicitly recorded values (standalone). Requires Python, pandas and matplotlib. No training or raw data loading is performed.
Save this notebook in your repository's notebooks/ directory to load its metric files automatically. Figures are exported to outputs/tchic/figures/, or figures/ beside a standalone run.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

# Locate the repository when this notebook is placed in its notebooks/ directory.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "outputs/tchic/runs").is_dir()), None)
FIGURE_DIR = (ROOT / "outputs/tchic/figures" if ROOT else Path.cwd() / "figures")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Exact metrics recorded in the development evaluation, not estimated from a figure.
recorded = {
    "Mean baseline": [0.3355354402, 0.1443537243, 0.6587230807, 0.4647533618, 0.6572280063, 0.4684528127],
    "BABEL · 2 epochs": [0.3673974223, 0.2091734982, 0.6418067781, 0.4635386101, 0.6383854283, 0.4604621979],
    "BABEL · 20 epochs": [0.3527711584, 0.1783944994, 0.6723915276, 0.4693196018, 0.6706889052, 0.4718005928],
}
columns = ["RMSE", "MAE", "mean_Pearson_cell", "mean_Spearman_cell",
           "overall_Pearson", "overall_Spearman"]
paths = ["babel_smoke/mean_baseline_metrics.txt", "babel_smoke/babel_metrics.txt",
         "babel_20epochs/babel_metrics.txt"]

def read_metrics(path):
    return {k.strip(): float(v.strip()) for k, v in
            (line.split(":", 1) for line in path.read_text().splitlines() if line.strip())}

if ROOT is not None:
    files = [ROOT / "outputs/tchic/runs" / p for p in paths]
    if not all(p.is_file() for p in files):
        raise FileNotFoundError("Repository found but some metric files are missing: "
                                + str([str(p) for p in files if not p.is_file()]))
    metrics = pd.DataFrame([read_metrics(p) for p in files], index=recorded.keys())[columns]
    print("Source: saved metric files in", ROOT)
else:
    metrics = pd.DataFrame.from_dict(recorded, orient="index", columns=columns)
    print("Source: exact recorded values supplied in the conversation (standalone mode).")

assert metrics.notna().all().all(), "Missing metrics"
print(metrics.round(6).to_string())


In [ ]:
import matplotlib.pyplot as plt

labels = ["Mean\nbaseline", "BABEL\n2 epochs", "BABEL\n20 epochs"]
colors = ["navy", "skyblue", "steelblue"]

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# Pearson correlation
bars = axes[0].bar(labels, metrics["mean_Pearson_cell"], color=colors)
axes[0].set_title("Mean cell-wise Pearson ↑")
axes[0].set_ylim(0, 0.8)
axes[0].bar_label(bars, fmt="%.3f", padding=4)

# Prediction error
bars = axes[1].bar(labels, metrics["RMSE"], color=colors)
axes[1].set_title("RMSE ↓")
axes[1].set_ylim(0, 0.45)
axes[1].bar_label(bars, fmt="%.3f", padding=4)

# Small layout improvements
for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_axisbelow(True)
    ax.grid(axis="y", alpha=0.2)

plt.tight_layout()

fig.savefig(FIGURE_DIR / "babel_preliminary_results.png", dpi=300)
fig.savefig(FIGURE_DIR / "babel_preliminary_results.svg")
plt.show()